In [26]:
import json
import glob
import pandas as pd
from pprint import pprint

In [ ]:
# test_dir = 'data/all_events_seperated'
# test_output = 'all_events_combined.json'

# def merge_JsonFiles(data_dir, output_file_name):
#     result = {}
#     file_paths = glob.glob(data_dir + '/*.json')

#     for file in file_paths:
#         with open(file, 'r') as infile:
#                 data = json.load(infile)
                
#                 result = result | data

#     with open(output_file_name, 'w') as output_file:
#         json.dump(result, output_file, indent=4)

# merge_JsonFiles(test_dir, test_output)

In [497]:
with open('/Users/macuser/projects/ufc_prediction/sample_data_5_events.json', 'r') as infile:
    sample_data = json.load(infile)

test_fight = sample_data['0f7210aa8d61af8d']['33e301872e2b5680']
test_fight.keys()

dict_keys(['results', 'fight_totals', 'totals_per_round', 'sig_str', 'sig_str_per_round', 'fight_name'])

In [119]:
def split_combined_data_dicts(dict_to_split):

    split_data_dict = {}

    # test_fight_stats['sig_str_per_round']
    for key, value_list in dict_to_split.items():
        for i, val in enumerate(value_list):

            ## create new key based on inext
            new_key = f"{key}_{chr(97 + i)}"
            split_data_dict[new_key] = val

    return split_data_dict

def seperate_composite_columns(comp_dict):

    composite_free_dict = {}

    for key, val in comp_dict.items():

        if len(val.split(" of ")) == 2:

            landed_val = val.split(" of ")[0]
            attempted_val = val.split(" of ")[1]
            composite_free_dict['landed_' + key] = landed_val
            composite_free_dict['attempted_' + key] = attempted_val

        else:

            composite_free_dict[key] = val

    return composite_free_dict


def process_percentages(dict):

    for key in dict.keys():

        if 'percentage' in key:

            dict[key] = dict[key].replace("%", "").replace("---", "0")

    return dict

def process_per_round_dict(per_round_dict):

    split_dict = {}

    for round in per_round_dict.keys():

        split_dict[round] = process_percentages(seperate_composite_columns(split_combined_data_dicts(per_round_dict[round])))

    return split_dict

In [206]:
test_fight.keys()

def process_fight(raw_fight):

    # raw_fight = {}

    if 'error' in raw_fight:
        error_dict = {}

        return {"ERROR" : "Drop entry"}



    raw_fight['fight_totals_split'] = process_percentages(seperate_composite_columns(split_combined_data_dicts(raw_fight['fight_totals'])))
    raw_fight['totals_per_round_split'] = process_per_round_dict(raw_fight['totals_per_round'])
    raw_fight['sig_str_split'] = process_percentages(seperate_composite_columns(split_combined_data_dicts(raw_fight['sig_str'])))
    raw_fight['sig_str_per_round_split'] = process_per_round_dict(raw_fight['sig_str_per_round'])

    ## only select
    keep_keys = ['fight_name','results','fight_totals_split', 'totals_per_round_split', 'sig_str_split', 'sig_str_per_round_split']

    # Standard comprehension
    keep_dict = {k: raw_fight[k] for k in keep_keys if k in raw_fight}
    return keep_dict

def process_fights_in_event(raw_event):

    processed_event = {}

    for key, val in raw_event.items():

        ## every va
        if key in ['event_details', 'FUTURE EVENT']:

            processed_event[key] = val

        else:
            processed_val = process_fight(val)
            processed_event[key] = processed_val
        

    return processed_event

fight_clean_test = process_fight(test_fight)
# pprint(fight_clean_test)

test_event = sample_data['0f7210aa8d61af8d']

process_fights_in_event(test_event).values()
test_event.values()


dict_values([{'date': 'March 29, 2025', 'location': 'Mexico City, Distrito Federal, Mexico'}, {'results': {'method': 'Decision - Unanimous', 'round': '5', 'time': '5:00', 'time format': '5 Rnd (5-5-5-5-5)', 'referee': 'Herb Dean', 'details:': "Mike Bell 46 - 49.Sal D'amato 46 - 49.Chris Lee 46 - 49."}, 'fight_totals': {'fighters': ['Brandon Moreno', 'Steve Erceg'], 'knockdowns': ['0', '0'], 'significant strikes': ['89 of 176', '116 of 279'], 'significant strike percentage': ['50%', '41%'], 'total strike': ['95 of 182', '119 of 282'], 'takedowns': ['1 of 5', '0 of 0'], 'takedown percentage': ['20%', '---'], 'submission attempts': ['0', '0'], 'reversals': ['0', '0'], 'control': ['1:25', '0:00']}, 'totals_per_round': {'Round 1': {'fighters': ['Brandon Moreno', 'Steve Erceg'], 'knockdowns': ['0', '0'], 'significant strikes': ['21 of 40', '25 of 68'], 'significant strike percentage': ['52%', '36%'], 'total strike': ['21 of 40', '25 of 68'], 'takedowns': ['0 of 0', '0 of 0'], 'takedown perce

In [207]:
process_fights_in_event(test_event)


{'event_details': {'date': 'March 29, 2025',
  'location': 'Mexico City, Distrito Federal, Mexico'},
 '33e301872e2b5680': {'fight_name': 'Brandon Moreno vs Steve Erceg',
  'results': {'method': 'Decision - Unanimous',
   'round': '5',
   'time': '5:00',
   'time format': '5 Rnd (5-5-5-5-5)',
   'referee': 'Herb Dean',
   'details:': "Mike Bell 46 - 49.Sal D'amato 46 - 49.Chris Lee 46 - 49."},
  'fight_totals_split': {'fighters_a': 'Brandon Moreno',
   'fighters_b': 'Steve Erceg',
   'knockdowns_a': '0',
   'knockdowns_b': '0',
   'landed_significant strikes_a': '89',
   'attempted_significant strikes_a': '176',
   'landed_significant strikes_b': '116',
   'attempted_significant strikes_b': '279',
   'significant strike percentage_a': '50',
   'significant strike percentage_b': '41',
   'landed_total strike_a': '95',
   'attempted_total strike_a': '182',
   'landed_total strike_b': '119',
   'attempted_total strike_b': '282',
   'landed_takedowns_a': '1',
   'attempted_takedowns_a': '5'

In [ ]:
with open('/Users/macuser/projects/ufc_prediction/sample_data_5_events.json', 'r') as infile:
    sample_data = json.load(infile)

test_fight = sample_data['0f7210aa8d61af8d']['33e301872e2b5680']
test_fight.keys()

In [498]:
test_dir = 'data/test_5_files'
test_output = 'sample_data_5_events.json'

def merge_JsonFiles(data_dir, output_file_name):
    result = {}
    file_paths = glob.glob(data_dir + '/*.json')

    for file in file_paths:
        with open(file, 'r') as infile:
                raw_event = json.load(infile)
                # print(list(raw_event.keys())[0])
                # print(type(raw_event))
                processed_event = {}

                for key, val in raw_event.items():
                     if val == 'FUTURE EVENT':
                          continue

                     processed_event[key] = process_fights_in_event(val)
                # processed_event = process_fights_in_event(raw_event[list(raw_event.keys())[0]])
                
                result = result | processed_event

    with open(output_file_name, 'w') as output_file:
        json.dump(result, output_file, indent=4)

merge_JsonFiles(test_dir, test_output)

In [499]:
with open('/Users/macuser/projects/ufc_prediction/sample_data_5_events.json', 'r') as infile:
    sample_data = json.load(infile)

test_fight = sample_data['0f7210aa8d61af8d']['33e301872e2b5680']
test_fight.keys()

dict_keys(['fight_name', 'results', 'fight_totals_split', 'totals_per_round_split', 'sig_str_split', 'sig_str_per_round_split'])

In [500]:
## convert fight dict into a pandas dataframe row
import pandas as pd

test_event['33e301872e2b5680'].keys()

dict_keys(['results', 'fight_totals', 'totals_per_round', 'sig_str', 'sig_str_per_round', 'fight_name', 'fight_totals_split', 'totals_per_round_split', 'sig_str_split', 'sig_str_per_round_split'])

In [214]:
test_event['33e301872e2b5680']['results']

{'method': 'Decision - Unanimous',
 'round': '5',
 'time': '5:00',
 'time format': '5 Rnd (5-5-5-5-5)',
 'referee': 'Herb Dean',
 'details:': "Mike Bell 46 - 49.Sal D'amato 46 - 49.Chris Lee 46 - 49."}

In [267]:
def reformat_per_round_dict(per_round_dict):

    reformatted_dict = {}

    for round_num, round_stats_dict in per_round_dict.items():

        for key, val in round_stats_dict.items():
            reformmatted_col_name = round_num + ' ' + key
            reformatted_dict[reformmatted_col_name] = val

    return reformatted_dict


def dataframe_row_from_fight(fight_dict):
    df_name = pd.DataFrame({'matchup':[fight_dict['fight_name']]})
    df_results = pd.DataFrame([fight_dict['results']])
    df_fight_totals = pd.DataFrame([fight_dict['fight_totals_split']])
    df_totals_per_round = pd.DataFrame([reformat_per_round_dict(fight_dict['totals_per_round_split'])])
    df_sig_str = pd.DataFrame([fight_dict['sig_str_split']])
    df_sig_str_per_round = pd.DataFrame([reformat_per_round_dict(fight_dict['sig_str_per_round_split'])])

    df_full = pd.concat([df_name, df_results, df_fight_totals, df_totals_per_round, df_sig_str, df_sig_str_per_round], axis = 1)
    return df_full

test_fight_1rd = test_event['8fc8d66ad0074ab0']
test_fight_5rd = test_event['33e301872e2b5680']

test_df_row_1rd = dataframe_row_from_fight(test_fight_1rd)
test_df_row_5rd = dataframe_row_from_fight(test_fight_5rd)

list(test_df_row_5rd.columns)

# test_event.keys()
    


# ## dictionaries that do not need to be split up
# df_name = pd.DataFrame([test_event['33e301872e2b5680']['fight_name']])
# df_results = pd.DataFrame([test_event['33e301872e2b5680']['results']])
# df_fight_totals = pd.DataFrame([test_event['33e301872e2b5680']['fight_totals_split']])
# df_totals_per_round = pd.DataFrame([reformat_per_round_dict(test_event['33e301872e2b5680']['totals_per_round_split'])])
# df_sig_str = pd.DataFrame([test_event['33e301872e2b5680']['sig_str_split']])
# df_sig_str_per_round = pd.DataFrame([reformat_per_round_dict(test_event['33e301872e2b5680']['sig_str_per_round_split'])])

# df_full = pd.concat([df_name, df_results, df_fight_totals, df_totals_per_round, df_sig_str, df_sig_str_per_round], axis = 1)
# # df_full


['matchup',
 'method',
 'round',
 'time',
 'time format',
 'referee',
 'details:',
 'fighters_a',
 'fighters_b',
 'knockdowns_a',
 'knockdowns_b',
 'landed_significant strikes_a',
 'attempted_significant strikes_a',
 'landed_significant strikes_b',
 'attempted_significant strikes_b',
 'significant strike percentage_a',
 'significant strike percentage_b',
 'landed_total strike_a',
 'attempted_total strike_a',
 'landed_total strike_b',
 'attempted_total strike_b',
 'landed_takedowns_a',
 'attempted_takedowns_a',
 'landed_takedowns_b',
 'attempted_takedowns_b',
 'takedown percentage_a',
 'takedown percentage_b',
 'submission attempts_a',
 'submission attempts_b',
 'reversals_a',
 'reversals_b',
 'control_a',
 'control_b',
 'Round 1 fighters_a',
 'Round 1 fighters_b',
 'Round 1 knockdowns_a',
 'Round 1 knockdowns_b',
 'Round 1 landed_significant strikes_a',
 'Round 1 attempted_significant strikes_a',
 'Round 1 landed_significant strikes_b',
 'Round 1 attempted_significant strikes_b',
 'Rou

In [445]:
df_cols_file = pd.read_csv('data_schema.csv')
# df_cols_file['col_name'].value_counts()

counts = df_cols_file['col_name'].value_counts()
dup_cols = counts[counts > 1].index.tolist()
dup_cols

## for a given fight, loop through the list of known duplicate column names,

def deduplicate_fight_df_row(fight_df_row, dup_cols):

    
    applicable_dup_cols = [col for col in dup_cols if col in fight_df_row.columns.tolist()]
    
    ## First, verify that the duplicate columns have the same value. If a duplicate column has different values, then we have a problem
    for dup_col in applicable_dup_cols:
        
        ## get both duplicate column rows for a given duplicate column name
        dup_df_val = fight_df_row[dup_col]

        if dup_df_val.iloc[0,0] == dup_df_val.iloc[0,1]:

            pass

        else:
            ## if the values are different, raise an error
            raise ValueError("duplicate columns have different values, there is a mismatch in the data")
    
    # Remove one of the duplicated columns
    fight_df_row_dedup = fight_df_row.loc[:,~fight_df_row.columns.duplicated()]

    return fight_df_row_dedup

df_fight_row_test = dataframe_row_from_fight(test_event['33e301872e2b5680'])
test_dedup_row = deduplicate_fight_df_row(df_fight_row_test, dup_cols)
print(test_dedup_row.columns.value_counts().sort_values(ascending = False))
test_dedup_row
    
# test_df_row_5rd_dedup = test_df_row_5rd.loc[:,~test_df_row_5rd.columns.duplicated()].copy()
# test_df_row_5rd_dedup[dup_cols]
# test_df_row_5rd_dedup.columns.tolist()

matchup                       1
knockdowns_a                  1
round                         1
time                          1
time format                   1
                             ..
Round 5 landed_ground_a       1
Round 5 attempted_ground_a    1
Round 5 landed_ground_b       1
Round 5 attempted_body_b      1
Round 5 attempted_ground_b    1
Name: count, Length: 307, dtype: int64


,matchup,method,round,time,time format,referee,details:,fighters_a,fighters_b,knockdowns_a,...,Round 5 landed_distance_b,Round 5 attempted_distance_b,Round 5 landed_clinch_a,Round 5 attempted_clinch_a,Round 5 landed_clinch_b,Round 5 attempted_clinch_b,Round 5 landed_ground_a,Round 5 attempted_ground_a,Round 5 landed_ground_b,Round 5 attempted_ground_b
0,Brandon Moreno vs Steve Erceg,Decision - Unanimous,5,5:00,5 Rnd (5-5-5-5-5),Herb Dean,Mike Bell 46 - 49.Sal D'amato 46 - 49.Chris Le...,Brandon Moreno,Steve Erceg,0,...,19,56,0,0,0,0,0,0,0,0


In [401]:
cols = pd.read_csv('data_schema_dedup_raw.csv')
cols['cols_clean'] = cols.index.str.replace("'","").str.lstrip()
# cols['cols_clean']
# cols_clean = cols['col'].str.replace("'","").str.lstrip()
cols['cols_clean'].to_csv('data_schema_dedup.csv', index=False)

In [407]:

# len(filtered_counts)
# print(list(filtered_counts.index))

In [503]:
## create the database schema
df_cols_dedup_file = pd.read_csv('data_schema_dedup.csv')
df_cols = df_cols_dedup_file['col_name'].tolist()
df_schema = pd.DataFrame(columns = df_cols)
# df_schema


def dataframe_from_event(event_dict, df_schema, dup_cols_list):

    event_df = df_schema.copy()

    for fight in event_dict:
        if fight != 'event_details':
            df_fight_row = dataframe_row_from_fight(event_dict[fight])
            df_fight_row_dedup = deduplicate_fight_df_row(df_fight_row, dup_cols_list)

            event_df = pd.concat([event_df, df_fight_row_dedup], ignore_index = True)


    event_df['event_date'] = event_dict['event_details']['date']
    event_df['event_location'] = event_dict['event_details']['location']

    return event_df


all_dat = df_schema.copy()
for event in sample_data:

    print(event)

    event_df = dataframe_from_event(sample_data[event], df_schema, dup_cols)

    all_dat = pd.concat([all_dat, event_df])

all_dat.head()
print(all_dat.shape)

0f7210aa8d61af8d
0cfbbfa0ba6d9855
00e11b5c8b7bfeeb
0e2c2daf11b5d8f2
0c01568b1ac77bff
(59, 309)
